# Donnons un coût à l’erreur

Comparer deux erreurs de classe avant d’ouvrir le calcul de la perte.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#choisir-la-perte).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Repartons de la règle initiale, avec ses biais nuls. A et C sont bien classées ; B reste annoncée comme une voix. Compter une erreur nous dit où le modèle échoue, mais peu de choses sur l’ampleur du désaccord.

| Réponse | Score voix | Score percussion | Décision |
| --- | --- | --- | --- |
| Voix gagne de justesse | 0,1 | 0 | Voix, incorrect |
| Voix écrase l’autre score | 4,5 | −4,5 | Voix, incorrect |

Ces deux lignes sont des alternatives pour B, pas deux nouveaux extraits. Nous voulons qu’une erreur très assurée reçoive le coût le plus élevé.

Appelons perte ce coût numérique. Chaque extrait possède ici une seule classe attendue. CrossEntropyLoss compare ses scores bruts aux indices de classes ; les scores sont flottants et les indices de type long, alias de int64.

La première ligne prépare le critère. reduction="none" demande de garder une perte par extrait, au lieu de les réunir immédiatement. La seconde ligne lui fournit les réponses et les annotations. Quel extrait devrait alors coûter le plus cher ?

| Extrait | Perte approximative | Interprétation |
| --- | --- | --- |
| A | 0,05 | La bonne classe est fortement préférée. |
| B | 9 | La mauvaise classe est fortement préférée. |
| C | 0,01 | La bonne classe est encore plus nettement préférée. |

Les alternatives pour B coûtent environ 0,74 et 9. L’erreur de classe est la même ; son degré d’assurance change son coût.

D’où vient cette différence ? La softmax convertit les scores d’une ligne en nombres positifs dont la somme vaut un. Elle calcule exp(score), puis divise chaque résultat par la somme des exponentielles. exp est la fonction exponentielle.

La perte retient la probabilité attribuée à la classe annotée et prend son opposé logarithmique : −ln(p). ln est le logarithme naturel. Une probabilité proche de un donne un coût faible ; une probabilité presque nulle donne un coût élevé.

Pour A, p = exp(1,5)/(exp(1,5)+exp(−1,5)) ≈ 0,95 ; son coût est donc proche de 0,05. Pour la vraie classe percussion de B, p vaut environ 0,00012 : le coût monte près de 9.

Le calcul softmax puis log ci-dessus sert à comprendre le résultat. Fournissez directement scores à CrossEntropyLoss : elle réalise le calcul logarithmique stable nécessaire. Lui fournir ces probabilités ferait un autre calcul.

Pour entraîner le modèle, réunissons les trois pertes. pertes.mean() produit leur moyenne : environ 3,02, un tenseur scalaire de forme (). La valeur élevée vient surtout de B. Nous avons maintenant un coût à réduire ; reste à savoir comment déplacer les poids.

| Objet | Valeur |
| --- | --- |
| Perte A | 0,048587 |
| Perte B | 9,000123 |
| Perte C | 0,006715 |
| Moyenne | 3,018476 |

Valeurs arrondies à six décimales pour comparer avec le programme complet.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Les biais sont revenus à zéro dans cette expérience indépendante. Les deux lignes supplémentaires décrivent deux réponses possibles pour B.

In [4]:
scores = torch.tensor([[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
reponses_B = torch.tensor([[0.1, 0.], [4.5, -4.5]])
cibles_B = torch.tensor([1, 1], dtype=torch.long)


In [5]:
observer('scores', scores)
observer('cibles', cibles)
observer('reponses_B', reponses_B)
observer('cibles_B', cibles_B)

scores forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]
reponses_B forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 0.0], [4.5, -4.5]]
cibles_B forme (2,) type torch.int64 requires_grad False
  valeurs : [1, 1]


### Calculer l’état suivant

none garde une valeur par ligne. Le même critère peut examiner les trois extraits ou les deux alternatives de B.

In [6]:
critere = nn.CrossEntropyLoss(reduction="none")
pertes = critere(scores, cibles)
couts_B = critere(reponses_B, cibles_B)


In [7]:
observer('pertes', pertes)
observer('couts_B', couts_B)

pertes forme (3,) type torch.float32 requires_grad False
  valeurs : [0.04858732968568802, 9.000123023986816, 0.006715348921716213]
couts_B forme (2,) type torch.float32 requires_grad False
  valeurs : [0.7443966865539551, 9.000123023986816]


### Calculer l’état suivant

mean fait la moyenne des trois pertes. item en extrait le nombre Python pour l’affichage.

In [8]:
moyenne = pertes.mean()


In [9]:
observer('moyenne', moyenne)

moyenne forme () type torch.float32 requires_grad False
  valeurs : 3.0184755325317383


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [10]:
print("pertes par extrait :", pertes)
print("deux reponses pour B :", couts_B)
print("moyenne :", moyenne.item())


pertes par extrait : tensor([4.8587e-02, 9.0001e+00, 6.7153e-03])
deux reponses pour B : tensor([0.7444, 9.0001])
moyenne : 3.0184755325317383


### Calculer l’état suivant

softmax compare les classes de chaque ligne. La case [0,0] est la probabilité voix de A ; log est le logarithme naturel.

In [11]:
probabilites = scores.softmax(dim=1)
probabilite_A = probabilites[0, 0]
cout_A = -torch.log(probabilite_A)


In [12]:
observer('probabilites', probabilites)
observer('probabilite_A', probabilite_A)
observer('cout_A', cout_A)

probabilites forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.9525741338729858, 0.04742587357759476], [0.9998766183853149, 0.00012339458044152707], [0.9933071732521057, 0.006692850962281227]]
probabilite_A forme () type torch.float32 requires_grad False
  valeurs : 0.9525741338729858
cout_A forme () type torch.float32 requires_grad False
  valeurs : 0.04858734458684921


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("probabilite voix A :", probabilite_A.item())
print("cout A verifie :", cout_A.item())

probabilite voix A : 0.9525741338729858
cout A verifie : 0.04858734458684921


In [14]:
assert bool(tuple(pertes.shape) == (3,) and tuple(moyenne.shape) == ()), 'choisir-la-perte/main: tuple(pertes.shape) == (3,) and tuple(moyenne.shape) == ()'
assert bool(torch.allclose(pertes, torch.tensor([0.04858733, 9.00012302, 0.00671535]), atol=1e-6)), 'choisir-la-perte/main: torch.allclose(pertes, torch.tensor([0.04858733, 9.00012302, 0.00671535]), atol=1e-6)'
assert bool(couts_B[1] > couts_B[0] and abs(couts_B[0].item() - 0.7443967) < 1e-6), 'choisir-la-perte/main: couts_B[1] > couts_B[0] and abs(couts_B[0].item() - 0.7443967) < 1e-6'
assert bool(torch.allclose(cout_A, pertes[0], atol=1e-6) and abs(moyenne.item() - 3.0184753) < 2e-6), 'choisir-la-perte/main: torch.allclose(cout_A, pertes[0], atol=1e-6) and abs(moyenne.item() - 3.0184753) < 2e-6'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Quelle ligne reçoit la plus grande perte individuelle : 0, 1 ou 2 ?

In [15]:
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_cibles = torch.tensor([0,1,0])
exo_pertes = torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles)
exo_indice = ...  # Remplacer les points par votre réponse.
if exo_indice is Ellipsis or exo_indice is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_indice == exo_pertes.argmax().item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La seconde ligne préfère fortement la classe voix alors que la cible demande percussion.

</details>

### Exercice 2

Avec reduction="none", quelle est la forme de la sortie pour ces trois extraits ?

In [16]:
exo_scores = torch.zeros(3,2)
exo_cibles = torch.tensor([0,1,0])
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Une perte par extrait reste disponible avant une éventuelle moyenne.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_cibles = torch.tensor([0,1,0])
exo_pertes = torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles)
exo_indice = 1
if exo_indice is Ellipsis or exo_indice is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_indice == exo_pertes.argmax().item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_scores = torch.zeros(3,2)
exo_cibles = torch.tensor([0,1,0])
exo_forme = (3,)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [19]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()